# One Hot Encoding

In [1]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('cars.csv')

In [4]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [9]:
df['brand'].value_counts()

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

In [7]:
df['brand'].nunique()  # unique brands count

32

In [11]:
df['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [16]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

# 1. One Hot Encoding using Pandas

In [92]:
pd.get_dummies(df, columns=['fuel','owner'])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


## k - 1  Encoding --> Removing Multi-colinearity

In [20]:
pd.get_dummies(df, columns=['fuel','owner',], drop_first = True)

# drop_first = True --->> Removes the one column from each columns dummies 

# Dummies of fuel column:
    # fuel_CNG,	fuel_Diesel, fuel_LPG, fuel_Petrol 
    # from which one column is removed 

# Dummies of owner column:
    # owner_First Owner, owner_Fourth & Above Owner, owner_Second Owner, owner_Test Drive Car, owner_Third Owner
    # From which 1 column is removed

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


# 2. Using Sklearn

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,0:4], df.iloc[:, -1], test_size = 0.2, random_state=42)

In [30]:
X_train.head()

,brand,km_driven,fuel,owner
6518,Tata,2560,Petrol,First Owner
6144,Honda,80000,Petrol,Second Owner
6381,Hyundai,150000,Diesel,Fourth & Above Owner
438,Maruti,120000,Diesel,Second Owner
5939,Maruti,25000,Petrol,First Owner


In [66]:
from sklearn.preprocessing import OneHotEncoder

In [69]:
ohe = OneHotEncoder(drop = 'first', sparse_output=False, dtype = np.int32)

In [70]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

In [71]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

In [72]:
X_train_new.shape

(6502, 7)

In [73]:
np.hstack((X_train[['brand','km_driven']].values, X_train_new)).shape

(6502, 9)

In [74]:
np.hstack((X_train[['brand','km_driven']].values, X_train_new))

array([['Tata', 2560, 0, ..., 0, 0, 0],
       ['Honda', 80000, 0, ..., 1, 0, 0],
       ['Hyundai', 150000, 1, ..., 0, 0, 0],
       ...,
       ['Hyundai', 35000, 0, ..., 0, 0, 0],
       ['Maruti', 27000, 1, ..., 0, 0, 0],
       ['Maruti', 70000, 0, ..., 1, 0, 0]], dtype=object)

## One Hot Encoding With Top Categories

In [84]:
counts = df['brand'].value_counts()

In [85]:
df['brand'].nunique()
threshold = 100

In [88]:
repl = counts[counts <= threshold].index

In [91]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon')).astype(int).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
7328,0,0,0,1,0,0,0,0,0,0,0,0,0
3101,0,0,0,0,0,0,1,0,0,0,0,0,0
346,0,0,0,0,1,0,0,0,0,0,0,0,0
5986,0,0,0,0,0,0,1,0,0,0,0,0,0
7921,0,0,0,0,0,0,1,0,0,0,0,0,0
